# ✈️ Taller 6: Agencia de Viajes Semillero — Agentes con LangChain

Bienvenido a la **Agencia de Viajes Semillero**. Nuestra agencia recibe llamadas todo el día con preguntas distintas:

> *"¿Qué tal el clima en Cuenca este fin de semana?"*
> *"Quiero reservar 5 días en Galápagos, ¿cuánto sale en sucres?"*
> *"Muéstrame las reservas a nombre de Ana López."*

Una sola persona no puede contestar todo eso a la velocidad que pide el negocio. Hoy construimos un **agente con LangChain** que combina tres tipos de herramientas:

1. **Tools prefabricadas** de `langchain-community` — Wikipedia, búsqueda web, calculadora.
2. **Tools propias que consumen APIs públicas** — clima (wttr.in) y conversión de moneda.
3. **Tools propias contra una base SQLite** — el sistema interno de reservas de la agencia.

### Lo que aprenderás hoy

| Pieza | Qué hace | Ejemplo en este taller |
|---|---|---|
| **Tools de la comunidad** | Importar herramientas listas | `WikipediaQueryRun`, `DuckDuckGoSearchRun` |
| **`@tool` con API externa** | Envolver una API REST como tool | `consultar_clima` → wttr.in |
| **`@tool` contra una BD** | Que el agente lea y escriba en SQLite | `crear_reserva`, `consultar_reservas_cliente` |
| **`create_agent` (LangChain 1.x)** | Agente con *function calling* nativo | API moderno, más confiable que ReAct texto |
| **Mini-frontend** | Visualizar el estado del sistema | Tabla en `ipywidgets` |

⏱️ **Tiempo estimado:** 60 minutos

> ⚠️ **Antes de empezar:** este taller usa `gemma4:31b-cloud`, un modelo de **Ollama Cloud**. Necesitas autenticarte **una sola vez** ejecutando en tu terminal: `ollama signin`. El comando abrirá tu navegador para vincular tu cuenta de Ollama. No necesitas `ollama pull` — los modelos cloud viven en los servidores de Ollama, no en tu PC.

## 0. Conexión a Ollama Cloud

A diferencia de los talleres anteriores que usaban modelos locales (`gemma4:e2b`), hoy usamos un modelo mucho más grande (`gemma4:31b-cloud`) que **no se descarga**: corre en los servidores de Ollama y le pegamos por red. Esto trae dos ventajas para un agente:

- **Más capacidad de razonamiento** — escoge mejor qué tool llamar y cuándo detenerse.
- **Soporta *function calling* nativo** — el modelo devuelve la llamada a la tool como JSON estructurado en lugar de texto plano. Más confiable, menos errores de parsing.

### Pasos

1. **Una sola vez por máquina**, abre tu terminal (no aquí en Jupyter) y ejecuta:

   ```
   ollama signin
   ```

   Esto abre tu navegador para vincular tu cuenta de Ollama. Después de aceptar, tu CLI queda autenticado.

2. **Verifica** que tu instalación ve el modelo cloud:

   ```
   ollama list
   ```

   Deberías ver `gemma4:31b-cloud` en la lista (o disponible para invocación).

3. **Si te equivocas** y olvidas el `signin`, el primer `invoke` desde Python fallará con un error tipo `401 Unauthorized` o `model not found`. No es un bug del notebook — es el recordatorio de autenticarte.

In [ ]:
# Librerias que vamos a usar a lo largo del taller
# Nota: 'ddgs' es el sucesor de 'duckduckgo-search'; langchain 1.x lo importa con ese nombre.
!pip install langchain langchain-community langchain-ollama wikipedia ddgs numexpr ipywidgets pandas requests langchain-ollama -q

print("\nInstalacion completa. Recuerda haber ejecutado 'ollama signin' en tu terminal.")

## 1. Ping al modelo cloud

Antes de construir nada, confirmamos que la autenticación funciona. Si esta celda falla con `401` o `model not found`, vuelve al paso anterior y ejecuta `ollama signin` en tu terminal.

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="gemma4:31b-cloud", temperature=0)

# Ping: si esto imprime algo, estas autenticado
respuesta = llm.invoke("Responde unicamente: 'Cloud conectado.' y nada mas.")
print(respuesta.content)

## 2. El sistema interno: SQLite

La agencia ya tiene su sistema de reservas en un archivo SQLite: **`agencia.db`** (debe venir junto a este notebook). Una sola tabla `reservas` con los campos: `id`, `cliente`, `destino`, `fecha_inicio`, `fecha_fin`, `presupuesto_usd`, `estado`, `creada_en`.

La celda siguiente **no crea** la tabla: la BD ya viene con reservas precargadas. Verifica que el archivo exista y lista por consola las reservas existentes. Más adelante, en la sección 11, vas a tener un **buscador interactivo** que consulta la BD en vivo.

> 💡 **Por qué SQLite y no Mongo/Postgres/etc.**: SQLite es un archivo, sin servicios extra, persistente entre runs del notebook. Es la forma más rápida de simular un sistema real.

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

DB_PATH = "agencia.db"

if not Path(DB_PATH).exists():
    raise FileNotFoundError(
        f"No se encuentra '{DB_PATH}' en el directorio actual.\n"
        f"Asegurate de que el archivo este junto a este notebook."
    )

with sqlite3.connect(DB_PATH) as c:
    df_reservas = pd.read_sql_query(
        "SELECT id, cliente, destino, fecha_inicio, fecha_fin, presupuesto_usd, estado FROM reservas ORDER BY id",
        c,
    )

print(f"BD encontrada: {Path(DB_PATH).resolve()}")
print(f"Reservas registradas: {len(df_reservas)}\n")
print(df_reservas.to_string(index=False))

## 3. Tools prefabricadas — la suite de la comunidad

`langchain-community` trae **decenas** de tools listas para usar. Las que nos interesan hoy:

| Tool | Para qué sirve | Necesita API key? |
|---|---|---|
| `WikipediaQueryRun` | Buscar artículos enciclopédicos | ❌ No |
| `DuckDuckGoSearchRun` | Búsqueda web general | ❌ No |
| `llm-math` (via `load_tools`) | Calculadora con razonamiento | ❌ No |

Usarlas es **una línea**: importas, instancias, y la tool ya tiene `name` + `description` definidos por sus autores. El agente las consume igual que las tuyas.

In [ ]:
from langchain_community.tools import WikipediaQueryRun, DuckDuckGoSearchRun
from langchain_community.utilities import WikipediaAPIWrapper, DuckDuckGoSearchAPIWrapper
from langchain_community.agent_toolkits.load_tools import load_tools  # en langchain 1.x se movio aqui

# 1. Wikipedia — limitamos a 1 resultado y 1000 chars para no saturar el prompt
wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=1000, lang="es"))

# 2. DuckDuckGo — busqueda web sin API key
busqueda = DuckDuckGoSearchRun(api_wrapper=DuckDuckGoSearchAPIWrapper(region="ec-es", max_results=3))

# 3. Calculadora — load_tools devuelve una lista, sacamos el primer (y unico) elemento
calculadora = load_tools(["llm-math"], llm=llm)[0]

tools_comunidad = [wiki, busqueda, calculadora]

print(f"{len(tools_comunidad)} tools prefabricadas cargadas:\n")
for t in tools_comunidad:
    print(f"  - {t.name}")
    print(f"      {t.description[:90]}...\n")

## 4. Tools propias con APIs públicas

Aquí el estudiante aprende el patrón clave: **envolver cualquier API REST como `@tool`**. Las dos APIs que usamos son gratuitas y sin registro:

| API | Para qué | URL |
|---|---|---|
| `wttr.in` | Clima actual + pronóstico de cualquier ciudad del mundo | `https://wttr.in/{ciudad}?format=j1` |
| `open.er-api.com` | Tipos de cambio actualizados desde USD | `https://open.er-api.com/v6/latest/USD` |

**Buenas prácticas que aplico abajo:**

- `timeout` en cada request — un agente bloqueado es un agente muerto.
- `try/except` que devuelve **string descriptivo** en lugar de excepción. El agente lee el string y razona; una excepción Python lo confunde.
- Docstring claro con qué hace, qué espera y qué devuelve. **Lo que vea el agente.**

In [ ]:
import requests
from langchain.tools import tool

@tool
def consultar_clima(ciudad: str) -> str:
    """Consulta el clima actual de una ciudad usando wttr.in. Acepta cualquier ciudad del mundo (ej: 'Cuenca', 'Tokyo', 'Buenos Aires'). Devuelve temperatura, descripcion del cielo y humedad."""
    try:
        r = requests.get(f"https://wttr.in/{ciudad}", params={"format": "j1"}, timeout=10)
        if r.status_code != 200:
            return f"No se pudo consultar el clima de {ciudad} (codigo {r.status_code})."
        data = r.json()
        actual = data["current_condition"][0]
        temp_c = actual["temp_C"]
        desc = actual["lang_es"][0]["value"] if "lang_es" in actual else actual["weatherDesc"][0]["value"]
        humedad = actual["humidity"]
        return f"Clima en {ciudad}: {desc}, {temp_c} grados C, humedad {humedad}%."
    except Exception as e:
        return f"Error consultando clima de {ciudad}: {e}"

@tool
def convertir_usd_a(monto_usd: float, codigo_moneda: str) -> str:
    """Convierte un monto en dolares (USD) a otra moneda usando tipos de cambio actualizados. El codigo de moneda debe ser ISO 4217 (ej: 'EUR', 'ARS', 'COP', 'PEN', 'CLP', 'MXN'). Devuelve el monto convertido."""
    try:
        r = requests.get("https://open.er-api.com/v6/latest/USD", timeout=10)
        if r.status_code != 200:
            return f"No se pudo obtener tipo de cambio (codigo {r.status_code})."
        data = r.json()
        codigo = codigo_moneda.strip().upper()
        if codigo not in data["rates"]:
            return f"Moneda '{codigo}' no encontrada. Usa un codigo ISO 4217 valido."
        tasa = data["rates"][codigo]
        convertido = monto_usd * tasa
        return f"{monto_usd} USD = {convertido:.2f} {codigo} (tasa: 1 USD = {tasa} {codigo})."
    except Exception as e:
        return f"Error convirtiendo moneda: {e}"

tools_apis = [consultar_clima, convertir_usd_a]

# Prueba rapida fuera del agente
print(consultar_clima.invoke({"ciudad": "Cuenca"}))
print()
print(convertir_usd_a.invoke({"monto_usd": 100.0, "codigo_moneda": "EUR"}))

## 5. Tools propias contra la base de datos

Aquí es donde el agente deja de ser un consultor y pasa a ser un **operador**: lee y **escribe** en el sistema real de la agencia.

Las 6 tools cubren las acciones típicas de una agencia:

| Tool | Operación SQL | Para qué la usa el agente |
|---|---|---|
| `consultar_reservas_cliente` | `SELECT WHERE cliente LIKE` | "¿Qué reservas tiene Ana?" |
| `verificar_disponibilidad` | `SELECT WHERE destino AND fechas se solapan` | "¿Puedo ir a Cuenca esa semana?" |
| `crear_reserva` | `INSERT` | "Resérvame 5 días en Galápagos" |
| `actualizar_estado_reserva` | `UPDATE` | "Confirma la reserva #3" |
| `listar_destinos_populares` | `SELECT destino, COUNT(*) GROUP BY` | "¿Cuáles son los destinos más pedidos?" |
| `listar_reservas_por_estado` | `SELECT ... ORDER BY estado` + groupby en Python | "Dame un resumen por estado de todas las reservas" |

</br>

> ⚠️ **Cuidado con las tools que escriben** — la slide lo dice claro: una tool destructiva mal expuesta es un accidente esperando. Por eso `actualizar_estado_reserva` solo permite cambiar el estado, no borrar; y no incluimos `eliminar_reserva` a propósito.

In [ ]:
import sqlite3
from datetime import datetime

def _conn():
    """Conexion fresca para cada llamada — SQLite es liviano y asi evitamos locks."""
    c = sqlite3.connect(DB_PATH)
    c.row_factory = sqlite3.Row
    return c

@tool
def consultar_reservas_cliente(cliente: str) -> str:
    """Devuelve todas las reservas registradas para un cliente. La busqueda es por coincidencia parcial del nombre (ej: 'Ana' encuentra 'Ana Lopez')."""
    with _conn() as c:
        filas = c.execute(
            "SELECT id, destino, fecha_inicio, fecha_fin, presupuesto_usd, estado FROM reservas WHERE cliente LIKE ? ORDER BY fecha_inicio",
            (f"%{cliente.strip()}%",)
        ).fetchall()
    if not filas:
        return f"No hay reservas para clientes que coincidan con '{cliente}'."
    return f"Reservas para '{cliente}':\n" + "\n".join(
        f"  #{r['id']} | {r['destino']} | {r['fecha_inicio']} -> {r['fecha_fin']} | ${r['presupuesto_usd']:.2f} USD | {r['estado']}"
        for r in filas
    )

@tool
def verificar_disponibilidad(destino: str, fecha_inicio: str, fecha_fin: str) -> str:
    """Verifica si hay reservas existentes en el mismo destino cuyas fechas se solapan con el rango pedido. Fechas en formato YYYY-MM-DD. Devuelve 'disponible' o el detalle de los conflictos encontrados."""
    with _conn() as c:
        filas = c.execute(
            """SELECT id, cliente, fecha_inicio, fecha_fin FROM reservas
               WHERE destino = ? AND estado != 'cancelada'
                 AND NOT (fecha_fin < ? OR fecha_inicio > ?)""",
            (destino.strip(), fecha_inicio, fecha_fin)
        ).fetchall()
    if not filas:
        return f"Destino '{destino}' disponible entre {fecha_inicio} y {fecha_fin}."
    detalle = "; ".join(f"#{r['id']} ({r['cliente']}, {r['fecha_inicio']}->{r['fecha_fin']})" for r in filas)
    return f"Hay {len(filas)} reserva(s) en '{destino}' que solapan: {detalle}."

@tool
def crear_reserva(cliente: str, destino: str, fecha_inicio: str, fecha_fin: str, presupuesto_usd: float) -> str:
    """Crea una nueva reserva en estado 'pendiente'. Fechas en formato YYYY-MM-DD. Devuelve el ID de la reserva creada."""
    try:
        # Validacion liviana de fechas
        datetime.strptime(fecha_inicio, "%Y-%m-%d")
        datetime.strptime(fecha_fin, "%Y-%m-%d")
    except ValueError:
        return "Error: las fechas deben estar en formato YYYY-MM-DD."
    with _conn() as c:
        cur = c.execute(
            "INSERT INTO reservas (cliente, destino, fecha_inicio, fecha_fin, presupuesto_usd) VALUES (?, ?, ?, ?, ?)",
            (cliente.strip(), destino.strip(), fecha_inicio, fecha_fin, float(presupuesto_usd))
        )
        c.commit()
        nuevo_id = cur.lastrowid
    return f"Reserva creada con ID #{nuevo_id} para {cliente} a {destino} ({fecha_inicio} -> {fecha_fin}, ${presupuesto_usd:.2f} USD, estado: pendiente)."

@tool
def actualizar_estado_reserva(id_reserva: int, nuevo_estado: str) -> str:
    """Cambia el estado de una reserva existente. Estados validos: 'pendiente', 'confirmada', 'cancelada'."""
    estados_validos = {"pendiente", "confirmada", "cancelada"}
    if nuevo_estado not in estados_validos:
        return f"Estado invalido. Validos: {estados_validos}."
    with _conn() as c:
        cur = c.execute("UPDATE reservas SET estado = ? WHERE id = ?", (nuevo_estado, int(id_reserva)))
        c.commit()
        if cur.rowcount == 0:
            return f"No existe reserva con ID {id_reserva}."
    return f"Reserva #{id_reserva} ahora esta en estado '{nuevo_estado}'."

@tool
def listar_destinos_populares() -> str:
    """Devuelve el top 5 de destinos mas reservados, ordenados por cantidad de reservas."""
    with _conn() as c:
        filas = c.execute(
            "SELECT destino, COUNT(*) as total FROM reservas GROUP BY destino ORDER BY total DESC LIMIT 5"
        ).fetchall()
    if not filas:
        return "No hay reservas registradas todavia."
    return "Destinos mas reservados:\n" + "\n".join(f"  {r['destino']}: {r['total']} reserva(s)" for r in filas)

@tool
def listar_reservas_por_estado() -> str:
    """Devuelve todas las reservas agrupadas por su estado (pendiente, confirmada, cancelada). Para cada estado muestra el numero total y el detalle de cada reserva. Usar cuando el usuario pida un panorama general del sistema o un resumen por estado."""
    with _conn() as c:
        filas = c.execute(
            "SELECT id, cliente, destino, fecha_inicio, fecha_fin, presupuesto_usd, estado FROM reservas ORDER BY estado, fecha_inicio"
        ).fetchall()
    if not filas:
        return "No hay reservas registradas todavia."
    # Agrupamos en Python: SQLite no devuelve GROUP_CONCAT con structure tan facil
    grupos = {}
    for r in filas:
        grupos.setdefault(r["estado"], []).append(r)
    bloques = []
    for estado in sorted(grupos.keys()):
        reservas = grupos[estado]
        cabecera = f"{estado.upper()} ({len(reservas)} reserva(s)):"
        detalle = "\n".join(
            f"  #{r['id']} | {r['cliente']} | {r['destino']} | "
            f"{r['fecha_inicio']} -> {r['fecha_fin']} | ${r['presupuesto_usd']:.2f} USD"
            for r in reservas
        )
        bloques.append(cabecera + "\n" + detalle)
    return "Reservas agrupadas por estado:\n\n" + "\n\n".join(bloques)

tools_bd = [
    consultar_reservas_cliente,
    verificar_disponibilidad,
    crear_reserva,
    actualizar_estado_reserva,
    listar_destinos_populares,
    listar_reservas_por_estado,
]

# Prueba rapida fuera del agente
print(consultar_reservas_cliente.invoke({"cliente": "Ana"}))
print()
print(listar_reservas_por_estado.invoke({}))

## 6. Armando el agente — `create_agent`

Tenemos **10 tools en total** (3 prefabricadas + 2 con APIs + 5 con BD). En LangChain 1.x el API de agentes se simplificó: una sola función `create_agent` toma el modelo, las tools y un system prompt, y devuelve un agente listo para invocar.

> 💡 **Nota sobre versiones**: si vienes de tutoriales viejos verás `AgentExecutor` + `create_tool_calling_agent`. Esos siguen funcionando importándolos desde `langchain_classic.agents`, pero el camino oficial en 1.x es `create_agent` desde `langchain.agents`. Internamente usa LangGraph y maneja el bucle ReAct por nosotros.

1. **El system prompt** — le decimos al modelo quién es y cómo comportarse. Sin él el agente improvisa.
2. **El agente** — `create_agent(model, tools, system_prompt)`. Aprovecha que `gemma4:31b-cloud` soporta *function calling* nativo (las tool calls vienen como JSON estructurado, no como texto plano).

In [ ]:
from langchain.agents import create_agent

# Unimos las 3 familias de tools
todas_las_tools = tools_comunidad + tools_apis + tools_bd
print(f"Total de tools registradas: {len(todas_las_tools)}")
for t in todas_las_tools:
    print(f"  - {t.name}")

# System prompt: rol + reglas + estilo de respuesta
SYSTEM_PROMPT = """Eres el asistente virtual de la Agencia de Viajes Semillero.

Tu trabajo es atender consultas de clientes y gestionar reservas. Tienes herramientas para:
- Consultar el clima de cualquier ciudad.
- Convertir precios de USD a otras monedas.
- Buscar informacion turistica (Wikipedia, busqueda web).
- Hacer calculos numericos.
- Leer, crear y actualizar reservas en el sistema interno.

Reglas:
- Si el usuario quiere reservar, SIEMPRE verifica disponibilidad antes de crear la reserva.
- Cuando crees una reserva, confirma al usuario el ID asignado.
- Si te piden un precio en otra moneda, usa la herramienta de conversion (no inventes tasas).
- Las fechas en la base de datos van en formato YYYY-MM-DD.
- Si no tienes informacion suficiente, pregunta antes de actuar."""

agente = create_agent(
    model=llm,
    tools=todas_las_tools,
    system_prompt=SYSTEM_PROMPT,
)

# Helper para invocar al agente. Imprime de forma compacta cada tool ejecutada
# y la response que devolvio, terminando con la respuesta final del agente.
def consultar(pregunta: str, config=None):
    """Invoca al agente, muestra tools ejecutadas + responses, y devuelve la respuesta final."""
    print(f">>> Usuario: {pregunta}\n")
    resultado = agente.invoke(
        {"messages": [{"role": "user", "content": pregunta}]},
        config=config or {},
    )
    for m in resultado["messages"][1:]:  # saltamos el HumanMessage del usuario
        tipo = type(m).__name__
        if tipo == "AIMessage" and getattr(m, "tool_calls", None):
            for tc in m.tool_calls:
                print(f"  [TOOL]     {tc['name']}({tc['args']})")
        elif tipo == "ToolMessage":
            preview = (m.content or "").replace("\n", " ")[:300]
            print(f"  [RESPONSE] {m.name} -> {preview}")
    respuesta_final = resultado["messages"][-1].content
    print(f"\n>>> Agente:\n{respuesta_final}")
    return respuesta_final

print("\nAgente armado. consultar('pregunta') muestra cada tool y su response.")

## 7. Primera consulta — solo APIs externas

Empezamos suave: una pregunta que solo necesita las herramientas de clima y conversión. El helper `consultar()` va a imprimir, por cada paso, **`[TOOL] nombre(args)`** seguido de **`[RESPONSE] resultado`** — así puedes seguir exactamente qué eligió el agente y qué obtuvo de cada llamada.

In [ ]:
consultar("Cual es el clima actual en Quito y cuanto serian 500 USD convertidos a euros?")

## 8. Consulta compuesta — combinando todo

Ahora la prueba de fuego: una sola frase que obliga al agente a combinar **APIs externas + BD + cálculo**. El cliente quiere reservar Cuenca y necesita varias cosas en cascada:

1. Verificar **disponibilidad** en la BD (¿hay conflicto?).
2. Consultar **clima** (¿necesita paraguas?).
3. Convertir el **presupuesto** a moneda local.
4. **Crear** la reserva en la BD.

Después de esta celda, ve a la sección 11 (Chat con el agente) y usa el buscador de la izquierda para ver la nueva reserva.

In [ ]:
consultar(
    "El cliente Elena Vargas quiere reservar un viaje a Cartagena del 2026-09-05 al 2026-09-09 "
    "con un presupuesto de 1500 USD. Verifica disponibilidad, consulta el clima esperado, "
    "dame detalles un poco de historia de la ciudad, convierte el presupuesto a pesos"
)

print("\n>>> Subi al tablero de sección 3 y haz clic en 'Refrescar' para ver la nueva reserva.")

## 9. Consulta turística — Wikipedia y búsqueda

Una pregunta puramente informativa que activa las tools de Wikipedia y DuckDuckGo. Es lo que distingue a un agente de viajes útil de uno meramente transaccional: puede contarte de qué se trata el destino.

In [ ]:
consultar("Cuentame brevemente que puedo hacer en Banos, Ecuador, y que clima tendre alli hoy.")

## 10. Modificando el estado del sistema

Hasta aquí el agente leyó y creó. Ahora lo ponemos a **actualizar**: el cliente Bruno quiere cancelar su reserva.

Después de esta celda, ve a la sección 11 (Chat con el agente) y busca por "Bruno" en el panel izquierdo para ver el cambio.

In [ ]:
consultar("Bruno Cevallos quiere cancelar su reserva a Galapagos. Buscala y cambia su estado a cancelada.")

## 11. Chat interactivo + buscador de reservas

Llegamos a la pieza completa: una **mini-app de la agencia** dentro del notebook. La celda monta dos paneles lado a lado:

| 🗺️ Panel izquierdo | 💬 Panel derecho |
|---|---|
| Buscador de reservas | Chat con el agente |
| Escribes un cliente y clic en 🔍 Buscar → consulta la BD **en vivo** | Multi-turno con memoria (`InMemorySaver` + `thread_id`) |
| Cada clic abre una conexión fresca a SQLite | Cada respuesta muestra la secuencia de tools ejecutadas |

</br>

> 💡 **El buscador siempre consulta la BD en el momento del clic** — no hay caché. Si el agente acaba de crear una reserva en el chat de la derecha, basta con teclear el nombre y dar clic para verla aparecer del lado izquierdo.

**Prueba con esta secuencia para ver la magia del multi-turno + el refresh:**

1. *"¿Qué reservas tiene Ana?"* — chat
2. *"Crea una reserva para Diego Mora a Lima del 2026-10-10 al 2026-10-15 con 1200 USD"* — chat
3. **Escribe "Diego" en el buscador y clic en 🔍 Buscar** → aparece la nueva reserva
4. *"¿Cuánto sería ese presupuesto en soles peruanos?"* — chat (no menciona Lima ni 1200, el agente recuerda)
5. *"¿Qué clima estoy llevando para esas fechas?"* — chat (sigue recordando)

In [ ]:
import ipywidgets as widgets
import html as html_lib
import re
import sqlite3
import pandas as pd
from langgraph.checkpoint.memory import InMemorySaver

# ============================================================
# 1. Agente con memoria persistente (checkpointer + thread_id)
# ============================================================
memoria_chat = InMemorySaver()
agente_chat = create_agent(
    model=llm,
    tools=todas_las_tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=memoria_chat,
)
config_chat = {"configurable": {"thread_id": "chat-taller-6"}}
historial_chat = []  # historial visual; la memoria real vive en el checkpointer

#region Renderizacion Front HTML

# ============================================================
# 2. CSS unificado (clases con prefijo .sml-)
# El degradado #667eea -> #764ba2 se aplica a: headers de ambos paneles,
# botones, borde del input y burbujas del usuario.
# ============================================================
GRADIENT = "linear-gradient(135deg, #667eea 0%, #764ba2 100%)"
CSS_APP = f"""
<style>
.sml-app * {{ box-sizing: border-box; }}
.sml-card {{
    background: white; border-radius: 14px; padding: 16px;
    box-shadow: 0 2px 10px rgba(0,0,0,0.06);
    font-family: -apple-system, 'Segoe UI', system-ui, sans-serif;
}}
.sml-header {{
    display: flex; align-items: center; gap: 8px;
    color: white; background: {GRADIENT};
    padding: 12px 16px; margin: -16px -16px 14px -16px;
    border-radius: 14px 14px 0 0;
    font-weight: 600; font-size: 1.05em;
}}

/* ---------- Buscador (panel izquierdo): tabla ---------- */
.sml-search-wrap {{ max-height: 540px; overflow-y: auto; }}
.sml-table {{
    width: 100%; border-collapse: collapse; font-size: 0.86em;
    background: white;
}}
.sml-table thead th {{
    background: #f7f9fc; color: #555; text-align: left;
    text-transform: uppercase; font-size: 0.72em; letter-spacing: 0.5px;
    padding: 10px 8px; border-bottom: 2px solid #e0e4e8;
    position: sticky; top: 0; z-index: 1;
}}
.sml-table tbody tr {{
    border-bottom: 1px solid #f0f2f5;
    transition: background 0.12s ease;
}}
.sml-table tbody tr:hover {{ background: #f7f9fc; }}
.sml-table td {{ padding: 9px 8px; vertical-align: middle; }}
.sml-table .col-id {{ font-weight: 700; color: #667eea; }}
.sml-table .col-monto {{ font-weight: 600; color: #27ae60; }}
.sml-table .col-fechas {{ color: #555; font-size: 0.88em; }}
.sml-badge {{
    padding: 3px 9px; border-radius: 12px;
    font-size: 0.7em; font-weight: 700; text-transform: uppercase;
    letter-spacing: 0.3px; white-space: nowrap;
}}
.sml-badge-pendiente  {{ background: #fff3cd; color: #856404; }}
.sml-badge-confirmada {{ background: #d4edda; color: #155724; }}
.sml-badge-cancelada  {{ background: #f8d7da; color: #721c24; }}
.sml-vacio {{
    text-align: center; padding: 40px 20px; color: #95a5a6;
}}
.sml-vacio .icon {{ font-size: 2.5em; display: block; margin-bottom: 8px; opacity: 0.5; }}
.sml-resumen {{
    color: #666; font-size: 0.82em; margin-bottom: 10px;
    padding: 6px 10px; background: #f7f9fc; border-radius: 6px;
}}

/* ---------- Chat (panel derecho) ---------- */
/* Truco column-reverse: con flex-direction column-reverse el scroll natural
   queda anclado al fondo. Renderizamos los mensajes en orden inverso y
   visualmente aparecen normales (viejos arriba, nuevos abajo). */
.sml-chat-box {{
    height: 460px; overflow-y: auto;
    padding: 14px; border-radius: 10px;
    background: linear-gradient(180deg, #fafbfc 0%, #f0f4f8 100%);
    border: 1px solid #e0e4e8;
    display: flex; flex-direction: column-reverse;
}}
.sml-msg-row {{ display: flex; margin: 12px 0; animation: sml-in 0.25s ease-out; }}
@keyframes sml-in {{
    from {{ opacity: 0; transform: translateY(6px); }}
    to   {{ opacity: 1; transform: translateY(0); }}
}}
.sml-msg-row.user  {{ justify-content: flex-end; }}
.sml-msg-row.agent {{ justify-content: flex-start; }}
.sml-bubble {{
    max-width: 82%; padding: 11px 15px; border-radius: 16px;
    line-height: 1.5; word-wrap: break-word; font-size: 0.93em;
}}
.sml-bubble.user {{
    background: {GRADIENT};
    color: white; border-bottom-right-radius: 4px;
}}
.sml-bubble.agent {{
    background: white; color: #2c3e50;
    border: 1px solid #e0e4e8; border-bottom-left-radius: 4px;
    box-shadow: 0 1px 3px rgba(0,0,0,0.04);
}}
.sml-sender {{
    font-size: 0.72em; font-weight: 700; opacity: 0.75;
    margin-bottom: 5px; letter-spacing: 0.3px;
}}
.sml-trazas {{
    background: #fff8e1; border-left: 3px solid #f39c12;
    padding: 9px 12px; margin-bottom: 8px; border-radius: 6px;
    font-family: 'SF Mono', Consolas, monospace; font-size: 0.78em; color: #555;
}}
.sml-trazas-titulo {{
    display: flex; align-items: center; gap: 5px;
    font-weight: 700; color: #d68910; margin-bottom: 4px; font-size: 0.85em;
}}
.sml-trazas .accion {{ color: #2874a6; }}
.sml-trazas .obs    {{ color: #1e8449; }}
.sml-chat-vacio {{
    text-align: center; padding: 80px 20px; color: #95a5a6;
    margin: auto;  /* centrado dentro de flex column-reverse */
}}
.sml-chat-vacio .icon {{ font-size: 3em; display: block; margin-bottom: 10px; opacity: 0.4; }}

/* ---------- Botones con degradado y borde del input ---------- */
/* ipywidgets renderiza Button como <button class="jupyter-widgets jupyter-button">.
   Targeteamos con la clase custom que les ponemos via add_class(). */
.sml-btn-gradient button,
.sml-btn-gradient {{
    background: {GRADIENT} !important;
    color: white !important;
    border: none !important;
    font-weight: 600 !important;
    box-shadow: 0 2px 6px rgba(102,126,234,0.25) !important;
    transition: transform 0.1s ease, box-shadow 0.15s ease !important;
}}
.sml-btn-gradient button:hover,
.sml-btn-gradient:hover {{
    transform: translateY(-1px) !important;
    box-shadow: 0 4px 10px rgba(102,126,234,0.35) !important;
}}
.sml-btn-gradient-outline button,
.sml-btn-gradient-outline {{
    background: white !important;
    color: #667eea !important;
    border: 2px solid #667eea !important;
    font-weight: 600 !important;
}}
.sml-btn-gradient-outline button:hover,
.sml-btn-gradient-outline:hover {{
    background: #f7f9fc !important;
}}
/* Borde del Textarea con degradado: usamos un wrapper con background gradient
   + padding. La textarea por dentro queda con fondo blanco. */
.sml-input-gradient {{
    background: {GRADIENT};
    padding: 2px; border-radius: 10px;
    box-shadow: 0 2px 8px rgba(102,126,234,0.15);
}}
.sml-input-gradient textarea {{
    border: none !important; border-radius: 8px !important;
    background: white !important; padding: 8px 12px !important;
    font-family: inherit !important; font-size: 0.92em !important;
    outline: none !important;
}}
</style>
"""

# ============================================================
# 3. Buscador (panel izquierdo): consulta SIEMPRE fresca a la BD, tabla
# ============================================================
def buscar_html(filtro_cliente=""):
    """Abre una conexion nueva a SQLite y devuelve una tabla HTML. Sin cache."""
    conn = sqlite3.connect(DB_PATH)
    try:
        if filtro_cliente.strip():
            df = pd.read_sql_query(
                "SELECT id, cliente, destino, fecha_inicio, fecha_fin, presupuesto_usd, estado "
                "FROM reservas WHERE cliente LIKE ? ORDER BY id DESC",
                conn, params=(f"%{filtro_cliente.strip()}%",),
            )
        else:
            df = pd.read_sql_query(
                "SELECT id, cliente, destino, fecha_inicio, fecha_fin, presupuesto_usd, estado "
                "FROM reservas ORDER BY id DESC",
                conn,
            )
    finally:
        conn.close()  # cierre explicito: cada clic = consulta fresca, sin locks pendientes

    if df.empty:
        return ('<div class="sml-vacio"><span class="icon">🔍</span>'
                'No hay reservas que coincidan con tu busqueda.</div>')

    filas = []
    for _, r in df.iterrows():
        filas.append(f'''
        <tr>
          <td class="col-id">#{r['id']}</td>
          <td>👤 {html_lib.escape(r['cliente'])}</td>
          <td>📍 {html_lib.escape(r['destino'])}</td>
          <td class="col-fechas">📅 {r['fecha_inicio']} → {r['fecha_fin']}</td>
          <td class="col-monto">${r['presupuesto_usd']:.2f}</td>
          <td><span class="sml-badge sml-badge-{r['estado']}">{r['estado']}</span></td>
        </tr>''')

    return (
        f'<div class="sml-resumen">Mostrando <b>{len(df)}</b> reserva(s)</div>'
        '<table class="sml-table">'
        '<thead><tr>'
        '<th>ID</th><th>Cliente</th><th>Destino</th><th>Fechas</th><th>USD</th><th>Estado</th>'
        '</tr></thead>'
        f'<tbody>{"".join(filas)}</tbody>'
        '</table>'
    )

input_filtro = widgets.Text(
    value="", placeholder="👤 Cliente...",
    layout=widgets.Layout(width="auto", flex="1"),
)
btn_buscar = widgets.Button(description="Buscar", icon="search",
                            layout=widgets.Layout(width="auto"))
btn_todas  = widgets.Button(description="Ver todas", icon="list",
                            layout=widgets.Layout(width="auto"))
btn_buscar.add_class("sml-btn-gradient")
btn_todas.add_class("sml-btn-gradient-outline")

resultados_busqueda = widgets.HTML(value=f'<div class="sml-search-wrap">{buscar_html("")}</div>')

def hacer_busqueda(_=None):
    # Consulta fresca cada vez que se invoca (clic en Buscar o Enter en input)
    resultados_busqueda.value = f'<div class="sml-search-wrap">{buscar_html(input_filtro.value)}</div>'

def buscar_todas(_):
    input_filtro.value = ""
    hacer_busqueda()

btn_buscar.on_click(hacer_busqueda)
btn_todas.on_click(buscar_todas)
input_filtro.observe(lambda c: hacer_busqueda() if c["name"] == "value" else None, names="value")

panel_busqueda = widgets.VBox([
    widgets.HTML('<div class="sml-card sml-app"><div class="sml-header">🗺️ Buscador de reservas</div>'),
    widgets.HBox([input_filtro, btn_buscar, btn_todas],
                 layout=widgets.Layout(margin="0 0 10px 0")),
    resultados_busqueda,
    widgets.HTML('</div>'),
], layout=widgets.Layout(width="42%", padding="0 8px 0 0"))

# ============================================================
# 4. Chat (panel derecho): multi-turno con memoria + secuencia de tools
# ============================================================
def _md_a_html(texto):
    safe = html_lib.escape(texto)
    safe = re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", safe)
    safe = re.sub(r"`([^`]+)`", r"<code>\1</code>", safe)
    return safe.replace("\n", "<br>")

def _render_chat():
    if not historial_chat:
        return ('<div class="sml-chat-box"><div class="sml-chat-vacio">'
                '<span class="icon">✈️</span>Empieza una conversacion con el agente.</div></div>')
    # Iteramos en REVERSE porque el contenedor usa flex-direction: column-reverse.
    # DOM[0] = ultimo mensaje, queda visualmente al FONDO con scroll anclado alli.
    partes = ['<div class="sml-chat-box">']
    for msg in reversed(historial_chat):
        if msg["rol"] == "user":
            partes.append(
                f'<div class="sml-msg-row user"><div class="sml-bubble user">'
                f'<div class="sml-sender">👤 TU</div>{html_lib.escape(msg["contenido"])}'
                f'</div></div>'
            )
        else:
            partes.append('<div class="sml-msg-row agent"><div class="sml-bubble agent">')
            partes.append('<div class="sml-sender">🤖 AGENTE</div>')
            if msg.get("trazas"):
                partes.append('<div class="sml-trazas">')
                partes.append('<div class="sml-trazas-titulo">⚙️ Secuencia de tools</div>')
                for paso in msg["trazas"]:
                    if paso["tipo"] == "accion":
                        partes.append(
                            f'<div class="accion">→ {html_lib.escape(paso["tool"])}({html_lib.escape(str(paso["args"]))})</div>'
                        )
                    else:
                        partes.append(
                            f'<div class="obs">✓ {html_lib.escape(paso["tool"])} → {html_lib.escape(paso["preview"])}</div>'
                        )
                partes.append('</div>')
            partes.append(f'<div>{_md_a_html(msg["contenido"])}</div></div></div>')
    partes.append('</div>')
    return "".join(partes)

chat_box      = widgets.HTML(value=_render_chat())
input_chat    = widgets.Textarea(
    placeholder="Escribe tu pregunta al agente...",
    layout=widgets.Layout(width="100%", height="60px"),
)
input_chat.add_class("sml-input-gradient")  # wrapper con borde-degradado

btn_enviar    = widgets.Button(description="Enviar", icon="paper-plane",
                               layout=widgets.Layout(width="auto"))
btn_reiniciar = widgets.Button(description="Limpiar", icon="trash",
                               layout=widgets.Layout(width="auto"))
btn_enviar.add_class("sml-btn-gradient")
btn_reiniciar.add_class("sml-btn-gradient-outline")
estado_chat   = widgets.HTML(value="")

def _enviar(_=None):
    pregunta = input_chat.value.strip()
    if not pregunta:
        return
    btn_enviar.disabled = True
    estado_chat.value = '<i style="color:#666;font-size:0.85em;">⏳ El agente esta pensando...</i>'
    input_chat.value = ""
    historial_chat.append({"rol": "user", "contenido": pregunta})
    chat_box.value = _render_chat()
    # Try/except/finally: si cualquier paso falla (invoke, extraccion, render),
    # el finally GARANTIZA que el boton se re-habilita y el spinner se quita.
    try:
        # checkpointer + thread_id: solo enviamos el mensaje nuevo, el estado se guarda
        resultado = agente_chat.invoke(
            {"messages": [{"role": "user", "content": pregunta}]},
            config=config_chat,
        )
        msgs = resultado["messages"]
        # default=-1 evita ValueError si por algun motivo no hubiera HumanMessages
        idx = max((i for i, m in enumerate(msgs) if type(m).__name__ == "HumanMessage"), default=-1)
        trazas, respuesta_final = [], ""
        for m in msgs[idx + 1:]:
            tipo = type(m).__name__
            if tipo == "AIMessage" and getattr(m, "tool_calls", None):
                for tc in m.tool_calls:
                    trazas.append({"tipo": "accion", "tool": tc["name"], "args": tc["args"]})
            elif tipo == "ToolMessage":
                preview = (m.content or "").replace("\n", " ")[:250]
                trazas.append({"tipo": "obs", "tool": m.name, "preview": preview})
            elif tipo == "AIMessage" and m.content:
                respuesta_final = m.content
        if not respuesta_final:
            respuesta_final = "_(El agente no genero una respuesta final para este turno.)_"
        historial_chat.append({"rol": "agent", "contenido": respuesta_final, "trazas": trazas})
        chat_box.value = _render_chat()
    except Exception as e:
        import traceback
        err_msg = f"⚠️ **Error:** {type(e).__name__}: {e}"
        historial_chat.append({"rol": "agent", "contenido": err_msg, "trazas": []})
        chat_box.value = _render_chat()
        # Print al kernel log para depurar si el chat queda raro
        print(f"[chat _enviar] {type(e).__name__}: {e}")
        traceback.print_exc()
    finally:
        # GARANTIZADO: siempre re-habilitamos el boton, pase lo que pase
        estado_chat.value = ""
        btn_enviar.disabled = False

def _reiniciar(_=None):
    global config_chat
    historial_chat.clear()
    config_chat = {"configurable": {"thread_id": f"chat-taller-6-{id(historial_chat)}"}}
    chat_box.value = _render_chat()

btn_enviar.on_click(_enviar)
btn_reiniciar.on_click(_reiniciar)

panel_chat = widgets.VBox([
    widgets.HTML('<div class="sml-card sml-app"><div class="sml-header chat">💬 Chat con el agente</div>'),
    chat_box,
    estado_chat,
    input_chat,
    widgets.HBox([btn_enviar, btn_reiniciar]),
    widgets.HTML('</div>'),
], layout=widgets.Layout(width="58%", padding="0 0 0 8px"))

# ============================================================
# 5. Composicion final: CSS + HBox con los dos paneles
# ============================================================
display(widgets.HTML(CSS_APP))  # CSS global (una sola vez)
display(widgets.HBox([panel_busqueda, panel_chat],
                     layout=widgets.Layout(width="100%", align_items="flex-start")))

#endregion

## 12. Tu turno

Tres retos progresivos para que extiendas el agente:

### Reto A — Una tool nueva contra la BD

Agrega una tool `eliminar_reserva_cancelada(id_reserva: int)` que **solo** permita borrar reservas que ya estén en estado `cancelada`. Esto demuestra el principio de la slide: las tools destructivas deben tener **salvaguardas internas**.

### Reto B — Una API pública nueva

Envuelve una nueva API pública como `@tool`. Algunas ideas sin API key:

- [REST Countries](https://restcountries.com/) — info de país (moneda, idioma, capital).
- [TimeAPI](https://www.timeapi.io/) — hora actual en una zona horaria.
- [Nominatim](https://nominatim.openstreetmap.org/) — geocodificación (ciudad → coordenadas).

> 💡 **Recuerda los 4 principios de diseño de tools**: nombre descriptivo, docstring claro, tipos explícitos, una responsabilidad.

In [ ]:
# Plantilla para el Reto A
@tool
def eliminar_reserva_cancelada(id_reserva: int) -> str:
    """[ESCRIBE AQUI: descripcion clara de que hace, condiciones, y que devuelve]"""
    # 1. Verifica que la reserva exista y este en estado 'cancelada'
    # 2. Si si, borrala. Si no, devuelve mensaje explicando por que no se borro.
    # 3. NUNCA debe borrar una reserva pendiente o confirmada.
    pass

# Descomenta cuando tu tool este lista:
# todas_las_tools_v2 = todas_las_tools + [eliminar_reserva_cancelada]
# agente_v2 = create_agent(model=llm, tools=todas_las_tools_v2, system_prompt=SYSTEM_PROMPT)
# r = agente_v2.invoke({"messages": [{"role": "user", "content": "[TU CONSULTA]"}]})
# print(r["messages"][-1].content)

## 📋 Resumen

| Concepto | Qué aprendimos |
|---|---|
| **`langchain-community`** | Importar tools listas: Wikipedia, búsqueda web, calculadora — sin reinventar la rueda |
| **`@tool` con `requests`** | Envolver cualquier API REST como una tool del agente |
| **`@tool` con SQLite** | Permitir al agente leer y escribir en un sistema real |
| **`create_agent` (LangChain 1.x)** | Function calling nativo, API moderno basado en LangGraph |
| **System prompt + reglas** | El comportamiento del agente vive en el prompt — no en el código |
| **Mini-frontend con ipywidgets** | Visualizar el estado del sistema sin salir del notebook |

### Las 3 familias de tools, lado a lado

| Familia | Cuándo conviene |
|---|---|
| **Prefabricadas (comunidad)** | Cuando ya existe (Wiki, búsqueda, GitHub, Slack, ...) |
| **Custom con API externa** | Cuando hay un servicio externo pero no hay tool oficial |
| **Custom contra BD** | Cuando el agente debe interactuar con TU sistema interno |

### Cuidados que NO debes saltarte

- ⏱️ **`timeout`** en todas las llamadas externas.
- 🔒 **No exponer tools destructivas** sin salvaguardas (estado válido, doble confirmación, etc.).
- 📝 **System prompt explícito** — el agente solo es tan bueno como sus instrucciones.
- 🔄 **`max_iterations`** — siempre, sin excepción.
- 🔍 **`debug=True`** en desarrollo, logs estructurados en producción.

### Preguntas guía

- ¿En qué tarea el agente eligió tools de las 3 familias en una sola corrida?
- ¿Qué pasaría si quitaras el system prompt? Pruébalo.
- ¿Cuál es la tool más "peligrosa" del taller y por qué? ¿Cómo la endurecerías para producción?
- ¿Vale la pena `gemma4:31b-cloud` vs un modelo local más pequeño? ¿En qué se notó la diferencia?

---

✈️ **La Agencia de Viajes Semillero ya tiene asistente. Próxima parada: tu propio agente.**